# 18 模型 (`core.models`)

`LogisticRegression` 封装（sklearn 兼容）。

In [ ]:
import os, sys
from pathlib import Path

_project = os.path.abspath(os.path.join(os.getcwd(), ".."))
if _project.replace("\\", "/") not in [p.replace("\\", "/") for p in sys.path]:
    sys.path.insert(0, _project)

import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd

from hscredit import init_setting
init_setting()


def load_demo_excel():
    roots = [Path.cwd(), Path.cwd() / "examples", Path.cwd().parent]
    for root in roots:
        for name in ("hengshucredit_yyp.xlsx", "hscredit_yyp.xlsx"):
            p = root / name
            if p.is_file():
                return pd.read_excel(p), p
    raise FileNotFoundError(
        "未找到演示数据：请将 hengshucredit_yyp.xlsx 或 hscredit_yyp.xlsx 放在 examples/ 目录。"
    )


df, DATA_PATH = load_demo_excel()
print("数据文件:", DATA_PATH, "形状:", df.shape)

if "MOB1" in df.columns:
    df = df.copy()
    df["target_demo"] = (pd.to_numeric(df["MOB1"], errors="coerce").fillna(-1) > 3).astype(int)
    target = "target_demo"
elif "FPD" in df.columns:
    df = df.copy()
    t = pd.to_numeric(df["FPD"], errors="coerce")
    df = df.loc[t.notna()].copy()
    df["target_demo"] = (t.loc[df.index] > 0).astype(int)
    target = "target_demo"
else:
    raise ValueError("需要 MOB1 或 FPD 列以构造二分类标签")

exclude = {"MOB1", "MOB2", "target_demo", "CURRENT_DPD"}
num_cols = [
    c for c in df.columns
    if c not in exclude and pd.api.types.is_numeric_dtype(df[c])
]
print("目标:", target, "坏样本率:", f"{df[target].mean():.2%}", "数值列数:", len(num_cols))


In [ ]:
from sklearn.model_selection import train_test_split
from hscredit import LogisticRegression

use = [c for c in num_cols if df[c].notna().sum() > 30][:8]
X = df[use].fillna(df[use].median())
y = df[target]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=0)
m = LogisticRegression(max_iter=200)
m.fit(X_train, y_train)
print('train acc', float((m.predict(X_train) == y_train).mean()))
print('test acc', float((m.predict(X_test) == y_test).mean()))
